# content的使用

举例:保存多模态数据


In [3]:
import base64
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

def encode_image(img_path):
    with open(img_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

base64_image = encode_image("image_test.png")

model = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

# ==========重点：用 OpenAI 兼容的 image_url 结构！==========
msg = HumanMessage(
    content=[
        {"type": "text", "text": "描述这张图片"},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"
            }
        }
    ]
)

response = model.invoke([msg])
print(response.content)


这张图片展示了一瓶**雅诗兰黛（Estée Lauder）的“Double Wear”系列粉底液**。

画面中的主体是一个长方形的透明玻璃瓶，瓶内盛有浅米色的液体粉底。瓶子顶部配有一个金色的按压式泵头，外部罩着一个透明的长方体塑料盖。瓶身正中央有一个金色的方形标签，上面清晰地印有品牌名称。

图片采用了简约的静物摄影风格，背景是温暖的浅米色或沙色，光线柔和，在瓶身右侧投射出清晰的阴影，突出了产品的质感和立体感。瓶身下方的文字显示了该产品的具体色号信息。


# 2.content_block使用
举例

In [4]:
import base64
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

def encode_image(img_path):
    with open(img_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

base64_image = encode_image("image_test.png")

model = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

# ==========重点：用 OpenAI 兼容的 image_url 结构！==========
msg = HumanMessage(
    #content=[
    #    {"type": "text", "text": "描述这张图片"},
    #    {
    #        "type": "image_url",
    #        "image_url": {
    #           "url": f"data:image/jpeg;base64,{base64_image}"
    #      }
    #   }
    # ]

    content_blocks=[
            {'type': 'text', 'text': '这张图里有什么？'},
            {
                'type': 'image',
                'base64': base64_image,
                'mime_type': 'image/png',
            }
    ]
)

response = model.invoke([msg])
print(response.content)


这张图片展示了一瓶**粉底液**，从外观特征来看，它属于**Bobbi Brown（芭比波朗）**品牌。

以下是图片中的详细内容：

1. **产品主体**：画面中央是一个长方体的透明玻璃瓶，里面装有浅米色的液体（粉底液）。
2. **瓶口设计**：瓶口配有金色的按压泵头，外面罩着一个透明的长方体塑料盖，盖子上方带有金色的装饰边。
3. **标签信息**：
   - 瓶身中间有一块金色的方形标签，上面印有黑色的品牌名称，可以辨认出是 **"BOBBI BROWN"**。
   - 瓶身底部印有黑色的文字，标注了产品信息，可以看到 **"SKIN FOUNDATION"**（粉底液）以及可能表示色号或容量的字样。
4. **背景与光影**：产品放置在一个带有柔和纹理的暖米色背景前。光线从侧面照射过来，在瓶身左侧投下了一道清晰的斜长阴影，营造出高级、温馨的视觉效果。


# 4、更丰富的初始化参数类型
## 4.1 字符串列表


In [1]:
#1.导入相关依赖
from langchain_core.prompts import ChatPromptTemplate

# 2.定义聊天提示词模版
chat_template = ChatPromptTemplate.from_messages(
  [
    "Hello, {name}!"  # 等价于 ("human", "Hello, {name}!")
    ]
)

# 3. 使用invoke执行
messages = chat_template.invoke({"name":"小谷AI"})

# 4.打印格式化后的聊天提示词模版内容
print(messages)

messages=[HumanMessage(content='Hello, 小谷AI!', additional_kwargs={}, response_metadata={})]


## 4.2 元组（tuple）列表

In [2]:
# 示例: 元组形式的消息
prompt = ChatPromptTemplate.from_messages([
    ("system", "你的名字是{role}."),
    ("human", "很高兴认识你"),
])

print(prompt.invoke({"role":"小智"}))

messages=[SystemMessage(content='你的名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})]


## 4.3 字典列表（dict）

In [3]:
# 示例: 字典形式的消息
prompt = ChatPromptTemplate.from_messages([
    {"role": "system", "content": "你的名字是{role}."},
    {"role": "human", "content":"很高兴认识你"},
])

print(prompt.invoke({"role":"小智"}))

messages=[SystemMessage(content='你的名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})]


## 4.4 消息对象列表

In [4]:
# 示例: 字典形式的消息
prompt = ChatPromptTemplate.from_messages([
    {"role": "system", "content": "你的名字是{role}."},
    {"role": "human", "content":"很高兴认识你"},
])

print(prompt.invoke({"role":"小智"}))

messages=[SystemMessage(content='你的名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})]


说明：消息对象中是否可以声明变量？